# Experiment 05 — Word and Sentence Construction

```
Language comprehension
      ↓
Meaning + memory + emotion + social context
      ↓
Response planning
      ↓
Word and sentence construction   <- this notebook
      ↓
Motor commands
```

Stage 04 handed off a `plan` — *what kind* of thing to say (answer directly, ask a
clarifying question, empathize, give an instruction). This stage turns a plan into
actual words, one word at a time — an RNN language model conditioned on which plan
it's supposed to be executing.

Standalone as always: the plan is just an index into 4 categories, generated
directly here rather than imported from stage 04's model.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0);

## Toy sentences, four per plan

Small, hand-written, and deliberately templated — enough for the network to learn
"this plan sounds like this," not enough to generalize to genuinely new sentences.
We'll be honest about that in the results.

In [2]:
sentences_by_plan = {
    "answer_directly": [
        "the meeting starts at three",
        "yes that is correct",
        "the answer is forty two",
        "it is on the table",
    ],
    "ask_clarifying_question": [
        "could you say that again",
        "which one do you mean",
        "can you be more specific",
        "did you mean today or tomorrow",
    ],
    "empathize": [
        "that sounds really hard",
        "i am sorry you feel that way",
        "it makes sense to be upset",
        "i hear you and that matters",
    ],
    "give_instruction": [
        "press the button twice",
        "turn left at the corner",
        "save the file first",
        "restart the device now",
    ],
}
plans = list(sentences_by_plan.keys())

vocab = {"<bos>": 0, "<eos>": 1}
for sents in sentences_by_plan.values():
    for s in sents:
        for w in s.split():
            vocab.setdefault(w, len(vocab))
id_to_word = {i: w for w, i in vocab.items()}

print(f"{len(plans)} plans, {sum(len(v) for v in sentences_by_plan.values())} sentences, vocab size {len(vocab)}")

4 plans, 16 sentences, vocab size 60


## Architecture: a plan-conditioned recurrent generator

At each timestep: embed the current word, concatenate it with the (fixed, for the
whole sentence) plan embedding, and feed that into a `GRUCell`. The hidden state
gets projected to a distribution over the vocabulary for the *next* word. Same idea
as Module 09's neural n-gram, but recurrent (unbounded context) and conditioned on
an external plan vector rather than just the preceding words.

In [3]:
class Generator(nn.Module):
    def __init__(self, vocab_size, n_plans, word_dim=16, plan_dim=8, hidden_dim=32):
        super().__init__()
        self.word_embed = nn.Embedding(vocab_size, word_dim)
        self.plan_embed = nn.Embedding(n_plans, plan_dim)
        self.cell = nn.GRUCell(word_dim + plan_dim, hidden_dim)
        self.to_vocab = nn.Linear(hidden_dim, vocab_size)
        self.hidden_dim = hidden_dim

    def forward_sequence(self, token_ids, plan_idx):
        """Teacher-forced next-token logits for one sentence. token_ids includes <bos>...<eos>."""
        plan_vec = self.plan_embed(torch.tensor(plan_idx)).unsqueeze(0)
        h = torch.zeros(1, self.hidden_dim)
        logits_list = []
        for t in range(len(token_ids) - 1):
            w = self.word_embed(torch.tensor(token_ids[t])).unsqueeze(0)
            x = torch.cat([w, plan_vec], dim=1)
            h = self.cell(x, h)
            logits_list.append(self.to_vocab(h).squeeze(0))
        return torch.stack(logits_list)

    def generate(self, plan_idx, max_len=10):
        plan_vec = self.plan_embed(torch.tensor(plan_idx)).unsqueeze(0)
        h = torch.zeros(1, self.hidden_dim)
        token = vocab["<bos>"]
        out_tokens = []
        for _ in range(max_len):
            w = self.word_embed(torch.tensor(token)).unsqueeze(0)
            x = torch.cat([w, plan_vec], dim=1)
            h = self.cell(x, h)
            logits = self.to_vocab(h).squeeze(0)
            token = logits.argmax().item()
            if token == vocab["<eos>"]:
                break
            out_tokens.append(token)
        return out_tokens


model = Generator(vocab_size=len(vocab), n_plans=len(plans))
print(model)

Generator(
  (word_embed): Embedding(60, 16)
  (plan_embed): Embedding(4, 8)
  (cell): GRUCell(24, 32)
  (to_vocab): Linear(in_features=32, out_features=60, bias=True)
)


## Training

Teacher forcing: sum the next-token cross-entropy over every sentence, every plan, each epoch.

In [4]:
def encode(sentence):
    return [vocab["<bos>"]] + [vocab[w] for w in sentence.split()] + [vocab["<eos>"]]


optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

losses = []
for epoch in range(400):
    total_loss = 0.0
    optimizer.zero_grad()
    for plan_idx, plan in enumerate(plans):
        for sentence in sentences_by_plan[plan]:
            token_ids = encode(sentence)
            logits = model.forward_sequence(token_ids, plan_idx)
            target = torch.tensor(token_ids[1:])
            total_loss = total_loss + F.cross_entropy(logits, target)
    total_loss.backward()
    optimizer.step()
    losses.append(total_loss.item())

print(f"summed loss over all 16 sentences: {losses[0]:.2f} -> {losses[-1]:.4f}")

summed loss over all 16 sentences: 66.76 -> 3.7787


## Generating from each plan

Greedy decoding (always pick the highest-probability next word) starting from
`<bos>`, once per plan.

In [5]:
for plan_idx, plan in enumerate(plans):
    tokens = model.generate(plan_idx)
    sentence = " ".join(id_to_word[t] for t in tokens)
    print(f"{plan:24} -> {sentence!r}")

answer_directly          -> 'the answer is forty two'
ask_clarifying_question  -> 'which one do you mean'
empathize                -> 'i hear you and that matters'
give_instruction         -> 'restart the device now'


## What this hands off (conceptually) — and an honest caveat

A word sequence, which stage 06 (`motor commands`) would turn into articulatory
trajectories. But look closely at the generated sentences: they're not new
compositions — every one is exactly one of the four memorized training sentences
for that plan, reproduced word for word. The summed loss (66.76 -> 3.78 over ~80
total token predictions across all 16 sentences, roughly 0.05 per token) confirms
the model is genuinely confident, not just close: it isn't composing a sentence
from the plan, it's recognizing which of 4 memorized sequences to replay. This is
the same lesson Module 18 already taught with NanoGPT: a model with enough
capacity relative to its data will happily memorize instead of generalize. Getting
genuine word-level composition (not just "which of 4 sentences fits this plan")
needs many more examples per plan than 4 — a real target for when this stage gets
expanded.